A KGX can be produced from dgipy either through a Graph (nx-graph) or a JSON. 

Producing a KGX through a Graph should be prioritized as it will be much easier to produce and build off in the future compared to a JSON


KGX has a particular format necessary, the specific can be found here: https://github.com/biolink/kgx/blob/master/specification/kgx-format.md

In summary:

nodes:
- should have a "id" element (string) as a CURIE (currently id's are NOT valued as a CURIE, as is necessary) (https://biolink.github.io/biolink-model/id/)
- should have a category element (list)

edges:
- subject (id of source node)
- predicate (relationship between subject and object node)
- object (id of the target node)

There also seem to be a few additional edge elements that aren't listed in the previously mentioned linked (likely having been an older version). Some other necessary elements are (but not necessarily limited to):
- subject_node
- subject_object
- knowledge_level
- agent_type

In order to adjust for this new format, a complete rewrite of initalize_network() will be necessary

Currently, this rewrite of this function uses kgx's NxGraph instead of nx.Graph to ensure compatibility

The following is a WIP script to demonstrate:
- Generating an interactions data table and network graph
- The validiation of network graph (to check whether network graph adheres to the KGX format)
- The successful conversion of the network graph to a KGX GraphSource

Currently the primary error in the test script is the lack of proper CURIE ids present in the data

In [2]:
# Imports
from dgipy import dgidb
from dgipy import network_graph as ng

from kgx.source import GraphSource
from kgx.transformer import Transformer
from kgx.validator import Validator
from kgx.graph.nx_graph import NxGraph

import pandas as pd
from typing import List

# Rough Implementation of initalize_network() (WIP)
def __initalize_network(interactions: pd.DataFrame, selected_genes: List) -> NxGraph:
    interactions_graph = NxGraph()
    graphed_genes = set()
    for index in interactions.index:
        graphed_genes.add(interactions["gene"][index])
        interactions_graph.add_node(interactions["gene"][index], id=interactions["gene"][index], category=["Placeholder"])
        interactions_graph.add_node(interactions["drug"][index], id=interactions["drug"][index], category=["Placeholder"])
        interactions_graph.add_edge(
            id=interactions["gene"][index] + " - " + interactions["drug"][index],
            subject_node=interactions["gene"][index],
            object_node=interactions["drug"][index],
            subject=interactions["gene"][index],
            object=interactions["drug"][index],
            predicate="Placeholder",
            knowledge_level="Placeholder",
            agent_type="Placeholder"
        )
    ungraphed_genes = set(selected_genes).difference(graphed_genes)
    for gene in ungraphed_genes:
        interactions_graph.add_node(gene, id=gene, category=["Placeholder"])
    return interactions_graph

# Generate Data
interactions = dgidb.get_interactions(["BRAF", "PDGFRA"])
network_graph = __initalize_network(interactions, ["BRAF", "PDGFRA"])

# Check initalize_networK() output 
print(network_graph.nodes())
print(network_graph.edges())

# Validate Data
validator = Validator(verbose=True)
validator.validate(network_graph)
errors = validator.get_errors()

if(len(errors) == 0):
    # Convert to KGX
    transformer = Transformer()
    source = GraphSource(transformer)
    g = source.parse(graph=network_graph)
else:
    print(errors)

[('PDGFRA', {'id': 'PDGFRA', 'category': ['Placeholder']}), ('QUIZARTINIB', {'id': 'QUIZARTINIB', 'category': ['Placeholder']}), ('ZM447439', {'id': 'ZM447439', 'category': ['Placeholder']}), ('TAK-593', {'id': 'TAK-593', 'category': ['Placeholder']}), ('PONATINIB', {'id': 'PONATINIB', 'category': ['Placeholder']}), ('IMATINIB', {'id': 'IMATINIB', 'category': ['Placeholder']}), ('DASATINIB ANHYDROUS', {'id': 'DASATINIB ANHYDROUS', 'category': ['Placeholder']}), ('RAMUCIRUMAB', {'id': 'RAMUCIRUMAB', 'category': ['Placeholder']}), ('BECAPLERMIN', {'id': 'BECAPLERMIN', 'category': ['Placeholder']}), ('MOTESANIB', {'id': 'MOTESANIB', 'category': ['Placeholder']}), ('SERALUTINIB', {'id': 'SERALUTINIB', 'category': ['Placeholder']}), ('DOVITINIB', {'id': 'DOVITINIB', 'category': ['Placeholder']}), ('BARASERTIB', {'id': 'BARASERTIB', 'category': ['Placeholder']}), ('PD-0166285', {'id': 'PD-0166285', 'category': ['Placeholder']}), ('NINTEDANIB ESYLATE', {'id': 'NINTEDANIB ESYLATE', 'category':